# 03 - Herramientas y Agentes

Este es el corazón del taller. Aprenderemos a:
- Crear herramientas personalizadas con `@tool`
- Construir un agente que decide cuándo usar cada herramienta
- Entender el ciclo de razonamiento del agente
- Darle memoria conversacional al agente

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="gemma3:12b", temperature=0)

## 3.1 Crear herramientas con `@tool`

Una herramienta es simplemente una función Python decorada con `@tool`. El docstring le dice al agente cuándo usarla.

In [ ]:
from langchain_core.tools import tool
import math
import datetime


@tool
def calculadora(expresion: str) -> str:
    """Evalúa una expresión matemática. Úsala cuando necesites hacer cálculos.
    Ejemplo de entrada: '2 + 2' o 'math.sqrt(16)'"""
    try:
        resultado = eval(expresion, {"math": math, "__builtins__": {}})
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error en el cálculo: {e}"


@tool
def fecha_actual() -> str:
    """Devuelve la fecha y hora actual. Úsala cuando pregunten qué día es hoy o la hora."""
    ahora = datetime.datetime.now()
    return ahora.strftime("Hoy es %A %d de %B de %Y, son las %H:%M")


@tool
def buscar_en_diccionario(palabra: str) -> str:
    """Busca la definición de una palabra en un diccionario local simplificado."""
    diccionario = {
        "agente": "Sistema de IA que usa un LLM para razonar y tomar acciones de forma autónoma.",
        "llm": "Large Language Model - modelo de lenguaje grande entrenado con grandes cantidades de texto.",
        "rag": "Retrieval Augmented Generation - técnica que combina búsqueda de documentos con generación de texto.",
        "embedding": "Representación numérica (vector) de texto que captura su significado semántico.",
        "prompt": "Instrucción o texto de entrada que se le da a un modelo de lenguaje.",
    }
    definicion = diccionario.get(palabra.lower())
    if definicion:
        return f"{palabra}: {definicion}"
    return f"No encontré '{palabra}' en el diccionario. Palabras disponibles: {', '.join(diccionario.keys())}"


# Probemos las herramientas directamente
print(calculadora.invoke("math.sqrt(144) * 3"))
print(fecha_actual.invoke(""))
print(buscar_en_diccionario.invoke("rag"))

## 3.2 Bind Tools: conectar herramientas al modelo

Antes de crear un agente, veamos cómo el modelo "sabe" que tiene herramientas disponibles.

In [ ]:
herramientas = [calculadora, fecha_actual, buscar_en_diccionario]

# Conectar herramientas al modelo
llm_con_herramientas = llm.bind_tools(herramientas)

# El modelo decide si usar una herramienta o responder directamente
respuesta = llm_con_herramientas.invoke("¿Cuánto es 25 al cuadrado?")

print("Contenido:", respuesta.content)
print("Tool calls:", respuesta.tool_calls)

## 3.3 Crear un Agente con LangGraph

Un agente es un loop: el LLM decide qué herramienta usar, ejecuta la herramienta, observa el resultado, y decide si necesita otra herramienta o si puede responder.

In [ ]:
from langgraph.prebuilt import create_react_agent

# Crear el agente ReAct (Reason + Act)
agente = create_react_agent(
    model=llm,
    tools=herramientas,
)

# Probar el agente
resultado = agente.invoke(
    {"messages": [("human", "¿Qué día es hoy y cuánto es la raíz cuadrada de 256?")]}
)

# Mostrar el flujo de mensajes
for msg in resultado["messages"]:
    tipo = msg.__class__.__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"\n[{tipo}] Quiero usar herramientas:")
        for tc in msg.tool_calls:
            print(f"  → {tc['name']}({tc['args']})")
    elif tipo == "ToolMessage":
        print(f"[{tipo}] Resultado: {msg.content[:100]}")
    else:
        print(f"\n[{tipo}] {msg.content[:200]}")

## 3.4 Agente con prompt de sistema personalizado

Podemos darle una personalidad y contexto al agente.

In [ ]:
prompt_sistema = """Eres un asistente de investigación. Tu trabajo es ayudar a responder 
preguntas usando las herramientas disponibles. 

Reglas:
- Siempre usa la calculadora para operaciones matemáticas, no calcules de memoria
- Si te preguntan por la fecha, usa la herramienta fecha_actual
- Responde en español
- Sé conciso pero preciso"""

agente_personalizado = create_react_agent(
    model=llm,
    tools=herramientas,
    prompt=prompt_sistema,
)

resultado = agente_personalizado.invoke(
    {"messages": [("human", "Si tengo 1500 documentos y cada uno toma 3.5 minutos en procesar, ¿cuántas horas necesito?")]}
)

# Mostrar solo la respuesta final
print(resultado["messages"][-1].content)

## 3.5 Herramientas con acceso a archivos

Creemos una herramienta más útil: leer archivos del sistema.

In [ ]:
import os


@tool
def listar_archivos(directorio: str) -> str:
    """Lista los archivos en un directorio. Usa esta herramienta para explorar carpetas."""
    try:
        archivos = os.listdir(directorio)
        if not archivos:
            return f"El directorio '{directorio}' está vacío."
        return f"Archivos en '{directorio}':\n" + "\n".join(f"  - {a}" for a in sorted(archivos))
    except FileNotFoundError:
        return f"El directorio '{directorio}' no existe."


@tool
def leer_archivo(ruta: str) -> str:
    """Lee el contenido de un archivo de texto. Úsala para leer archivos .txt, .py, .md, etc."""
    try:
        with open(ruta, 'r', encoding='utf-8') as f:
            contenido = f.read(2000)  # Limitar a 2000 caracteres
        return contenido
    except FileNotFoundError:
        return f"El archivo '{ruta}' no existe."
    except Exception as e:
        return f"Error al leer el archivo: {e}"


# Agente con herramientas de archivos
agente_archivos = create_react_agent(
    model=llm,
    tools=[listar_archivos, leer_archivo, calculadora],
    prompt="Eres un asistente que ayuda a explorar archivos y carpetas. Responde en español.",
)

# Probar: explorar el directorio actual
resultado = agente_archivos.invoke(
    {"messages": [("human", "¿Qué archivos hay en el directorio actual (.)?" )]}
)
print(resultado["messages"][-1].content)

## 3.6 Agente con memoria conversacional

Para que el agente recuerde lo que hemos hablado, usamos un checkpointer de memoria.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

memoria = MemorySaver()

agente_con_memoria = create_react_agent(
    model=llm,
    tools=herramientas,
    prompt="Eres un asistente amigable. Recuerdas lo que el usuario te ha dicho antes. Responde en español.",
    checkpointer=memoria,
)

# Configuración de la sesión (thread_id identifica la conversación)
config = {"configurable": {"thread_id": "mi-conversacion-1"}}

# Turno 1
r1 = agente_con_memoria.invoke(
    {"messages": [("human", "Hola, me llamo Carlos y trabajo en la UBPD")]},
    config=config,
)
print("Turno 1:", r1["messages"][-1].content)

# Turno 2 - el agente debería recordar el nombre
r2 = agente_con_memoria.invoke(
    {"messages": [("human", "¿Cómo me llamo y dónde trabajo?")]},
    config=config,
)
print("\nTurno 2:", r2["messages"][-1].content)

## Ejercicio

1. Crea una herramienta `@tool` que convierta temperaturas de Celsius a Fahrenheit y viceversa
2. Crea otra herramienta que cuente las palabras de un texto dado
3. Construye un agente con estas herramientas + la calculadora
4. Prueba: "Si la temperatura en Bogotá es 14°C, ¿cuánto es en Fahrenheit? Y si tengo un texto de 500 palabras y cada palabra ocupa 5 bytes, ¿cuántos KB son?"

In [ ]:
# Tu código aquí
